# Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import time
import tracemalloc as tm

In [ ]:
df = pd.read_csv('Datasets/vote1.csv')
df.head(3)

In [ ]:
len(df)

In [ ]:
df.info()

In [ ]:
df.head(3)

# Formulas

In [ ]:
def R2(y_data:np.array, y_pred:np.array):
  y_mean = np.mean(y_data)

  ss_tot = np.sum((y_data - y_mean)**2)
  ss_res = np.sum((y_data - y_pred)**2)
  r2 = 1 - (ss_res/ss_tot)

  return r2

In [ ]:
# Regresion lineal: y = mx + b

Fórmulas de regresión lineal multiple

$$a = \sum_{i=1}^{n}x_{1i}$$
$$A = \sum_{i=1}^{n}x_{1i}^2$$
$$b = \sum_{i=1}^{n}x_{2i}$$
$$B = \sum_{i=1}^{n}x_{2i}^2$$
$$c = \sum_{i=1}^{n}x_{1i}x_{2i}$$

$$d_1=\sum_{i=1}^{n}{y_i}$$
$$d_1=\sum_{i=1}^{n}{y_ix_i}$$
$$d_1=\sum_{i=1}^{n}{y_i}$$

Determinante:
$$D=n(AB-c^2)-a^2B+2abc-b^2A$$

$$\alpha_0=\frac{(AB-c^2)d_1+(bc-aB)d_2+(ac-bA)d_3}{D}$$
$$\alpha_1=\frac{(bc-aB)d_1+(nB-b^2)d_2+(ab-nc)d_3}{D}$$
$$\alpha_2=\frac{(ac-bA)d_1+(ab-nc)d_2+(nA-a^2)d_3}{D}$$

In [ ]:
class RegresionLineal2Vars():
  def __init__(self, y_data:np.array, x1_data:np.array, x2_data:np.array) -> None:
    self.n = len(y_data)
    self.x1_data = x1_data
    self.x2_data = x2_data
    self.y_data = y_data

  def fit(self) -> None:
    t_0 = time.time()
    tm.start()

    n = self.n

    a = np.sum(self.x1_data)
    A = np.sum(self.x1_data**2)
    b = np.sum(self.x2_data)
    B = np.sum(self.x2_data**2)
    c = np.sum(self.x1_data*self.x2_data)

    d1 = np.sum(self.y_data)
    d2 = np.sum(self.y_data*self.x1_data)
    d3 = np.sum(self.y_data*self.x2_data)

    D = n*(A*B - c**2) - a**2*B + 2*a*b*c - b**2*A

    p1 = A*B - c**2
    p2 = b*c - a*B
    p3 = a*c - b*A
    self.alpha0 = (p1*d1 + p2*d2 + p3*d3)/D
    p1 = b*c - a*B
    p2 = n*B - b**2
    p3 = a*b - n*c
    self.alpha1 = (p1*d1 + p2*d2 + p3*d3)/D
    p1 = a*c - b*A
    p2 = a*b - n*c
    p3 = n*A - a**2
    self.alpha2 = (p1*d1 + p2*d2 + p3*d3)/D

    self.total_memory, self.peak_memory = tm.get_traced_memory()
    tm.stop()
    t_f = time.time()
    self.time = t_f - t_0

  def return_params(self) -> list:
    return [self.alpha0, self.alpha1, self.alpha2]

  def name_vars(self, y_name:str, x_1_name:str, x_2_name:str) -> None:
    self.y_name = y_name
    self.x_1_name = x_1_name
    self.x_2_name = x_2_name

  def __str__(self) -> str:
    return f"{self.y_name} = {self.alpha0} + {self.alpha1}*{self.x_1_name} + {self.alpha2}*{self.x_2_name}"

  def print_efficiency_details(self) -> None:
    print("Tiempo de Aprendizaje: ", self.time, "s", sep="")
    print("Memoria Total: ", self.total_memory, "B", sep="")
    print("Memoria Maxima: ", self.peak_memory, "B", sep="")

  def predict(self, x1_data:np.array, x2_data:np.array) -> np.array:
    return self.alpha0*np.ones(len(x1_data)) + self.alpha1*x1_data + self.alpha2*x2_data

In [ ]:
class RegresionLineal2VarGradient():
  def __init__(self, y_data:np.array, x1_data:np.array, x2_data:np.array) -> None:
    self.n = len(y_data)
    self.x1_data = x1_data
    self.x2_data = x2_data
    self.y_data = y_data

    self.MSEs = []

  def end(self, t_0:float, alphas:tuple) -> None:
    self.alpha0 = alphas[0]
    self.alpha1 = alphas[1]
    self.alpha2 = alphas[2]

    self.total_memory, self.peak_memory = tm.get_traced_memory()
    tm.stop()
    t_f = time.time()
    self.time = t_f - t_0

  def fit(self, learning_rate:float, epochs:int, tolerance:float, start:tuple) -> None:
    alpha0 = start[0]
    alpha1 = start[1]
    alpha2 = start[2]

    t_0 = time.time()
    tm.start()

    for i in range(epochs):
      y_pred = alpha0 + alpha1*self.x1_data + alpha2*self.x2_data
      mse = np.mean((y_pred - self.y_data)**2)/2

      self.MSEs.append(mse)

      if len(self.MSEs) != 1:
        if (np.abs(self.MSEs[-2] - mse)) < tolerance:
          self.last_epoch = i
          self.end(t_0, (alpha0, alpha1, alpha2))
          return

      alpha0 -= learning_rate*np.sum(y_pred - self.y_data)/self.n
      alpha1 -= learning_rate*np.dot((y_pred - self.y_data), self.x1_data)/self.n
      alpha2 -= learning_rate*np.dot((y_pred - self.y_data), self.x2_data)/self.n

    self.last_epoch = epochs
    self.end(t_0, (alpha0, alpha1, alpha2))

  def return_params(self) -> list:
    return [self.alpha0, self.alpha1, self.alpha2]

  def name_vars(self, y_name:str, x_1_name:str, x_2_name:str) -> None:
    self.y_name = y_name
    self.x_1_name = x_1_name
    self.x_2_name = x_2_name

  def __str__(self) -> str:
    return f"{self.y_name} = {self.alpha0} + {self.alpha1}*{self.x_1_name} + {self.alpha2}*{self.x_2_name}"

  def print_efficiency_details(self) -> None:
    print("Tiempo de Aprendizaje: ", self.time, "s", sep="")
    print("Memoria Total: ", self.total_memory, "B", sep="")
    print("Memoria Maxima: ", self.peak_memory, "B", sep="")
    print("Numero de Epochs:", self.last_epoch, "Epochs")

  def predict(self, x1_data:np.array, x2_data:np.array) -> np.array:
    return self.alpha0*np.ones(len(x1_data)) + self.alpha1*x1_data + self.alpha2*x2_data

## Fórmulas de regresión lineal simple

Para el modelo de una sola variable $y=\alpha_0+\alpha_1x$, los estimadores que minimizan la suma de errores al cuadrado (mínimos cuadrados ordinarios) están dados por:

$$\alpha_1=\frac{n\sum_{i=1}^{n}x_iy_i-\left(\sum_{i=1}^{n}x_i\right)\left(\sum_{i=1}^{n}y_i\right)}{n\sum_{i=1}^{n}x_i^2-\left(\sum_{i=1}^{n}x_i\right)^2}$$

$$\alpha_0=\bar{y}-\alpha_1\bar{x}$$

In [ ]:
class RegresionLinealSimple():
  def __init__(self, y_data:np.array, x_data:np.array) -> None:
    self.n = len(y_data)

    self.sum_x = np.sum(x_data)
    self.sum_x2 = np.sum(x_data**2)
    self.sum_y = np.sum(y_data)
    self.sum_xy = np.sum(x_data*y_data)

    self.alpha1 = (self.n*self.sum_xy - self.sum_x*self.sum_y) / (self.n*self.sum_x2 - self.sum_x**2)
    self.alpha0 = (self.sum_y - self.alpha1*self.sum_x) / self.n

  def return_params(self) -> list:
    return [self.alpha0, self.alpha1]

  def name_vars(self, y_name:str, x_name:str) -> None:
    self.y_name = y_name
    self.x_name = x_name

  def __str__(self) -> str:
    return f"{self.y_name} = {self.alpha0} + {self.alpha1}*{self.x_name}"

  def predict(self, x_data:np.array) -> np.array:
    return self.alpha0*np.ones(len(x_data)) + self.alpha1*x_data

# Experimentos

## a.
$voteA=\alpha_0+\alpha_1*shareA$

Fórmulas Directas

In [ ]:
plt.figure(figsize=(5, 4))
sns.scatterplot(data=df, x="shareA", y="voteA")
plt.title("Scatter plot of shareA vs voteA")
plt.xlabel("shareA")
plt.ylabel("voteA")
plt.show()

In [ ]:
x_a = df["shareA"].to_numpy()
y_a = df["voteA"].to_numpy()

In [ ]:
model_a = RegresionLinealSimple(y_a, x_a)
model_a.name_vars("voteA", "shareA")

In [ ]:
plt.figure(figsize=(5, 4))
sns.scatterplot(x=x_a, y=y_a, label="Actual data")
x_line = np.linspace(x_a.min(), x_a.max(), 100)
plt.plot(x_line, model_a.predict(x_line), color="red", label="Regresión")
plt.title("voteA vs shareA")
plt.xlabel("shareA")
plt.ylabel("voteA")
plt.legend()
plt.show()

## b.
$voteA=\alpha_0+\alpha_1*expendA$

Fórmulas Directas

In [ ]:
plt.figure(figsize=(5, 4))
sns.scatterplot(data=df, x="expendA", y="voteA")
plt.title("Scatter plot of expendA vs voteA")
plt.xlabel("expendA")
plt.ylabel("voteA")
plt.show()

In [ ]:
x_b = df["expendA"].to_numpy()
y_b = df["voteA"].to_numpy()

In [ ]:
model_b = RegresionLinealSimple(y_b, x_b)
model_b.name_vars("voteA", "expendA")

In [ ]:
print(model_b)

In [ ]:
plt.figure(figsize=(5, 4))
sns.scatterplot(x=x_b, y=y_b, label="Actual data")
x_line = np.linspace(x_b.min(), x_b.max(), 100)
plt.plot(x_line, model_b.predict(x_line), color="red", label="Regresión")
plt.title("voteA vs expendA")
plt.xlabel("expendA")
plt.ylabel("voteA")
plt.legend()
plt.show()

In [ ]:
print("R^2:", R2(y_b, model_b.predict(x_b)))

¿Qué nos dicen los respectivos coeficientes $\alpha_0$, $\alpha_1$ y $\alpha_2$?
* $\alpha_0$: Es el valor esperado de la variable dependiente (voteA) cuando la variable independiente (expendA) es igual a cero.
* $\alpha_1$: Indica el cambio promedio en la variable dependiente (voteA) por cada unidad de aumento en la variable independiente (expendA).

Con respecto al archivo
* $\alpha_0$: Si un candidato no gasta nada de dinero (expendA = 0), se espera que obtenga aproximadamente un 51.75% de los votos.
* $\alpha_1$: Por cada unidad adicional de gasto, el porcentaje de votos aumenta en 0.0144 puntos porcentuales.

## c.
$voteA=\alpha_0+\alpha_1*shareA+\alpha_2*expendA$

Fórmulas Directas

In [ ]:
plt.figure(figsize=(5, 4))
sns.scatterplot(data=df, x="shareA", y="voteA")
plt.title("Scatter plot of shareA vs voteA")
plt.xlabel("shareA")
plt.ylabel("voteA")
plt.show()

In [ ]:
x_1 = df["shareA"].to_numpy()
x_2 = df["expendA"].to_numpy()
y = df["voteA"].to_numpy()

In [ ]:
model = RegresionLineal2Vars(y, x_1, x_2)
model.fit()
model.name_vars("voteA", "shareA", "expendA")

In [ ]:
print(model)

$voteA = 41.2572 + 0.2714*shareA + 0*expendB$

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(x_1, x_2, y, c='blue', marker='o', label='Actual data')
x1_surf, x2_surf = np.meshgrid(np.linspace(x_1.min(), x_1.max(), 100), np.linspace(x_2.min(), x_2.max(), 100))
y_surf = model.alpha0 + model.alpha1 * x1_surf + model.alpha2 * x2_surf
ax.plot_surface(x1_surf, x2_surf, y_surf, color='red', alpha=0.5, label='Regression Plane')

ax.set_xlabel('shareA')
ax.set_ylabel('expendA')
ax.set_zlabel('voteA')
ax.set_title('voteA vs shareA and expendA')

ax.legend()
plt.show()

In [ ]:
print("R^2:", R2(y, model.predict(x_1, x_2)))

In [ ]:
model.print_efficiency_details()

¿Qué nos dicen los respectivos coeficientes $\alpha_0$, $\alpha_1$ y $\alpha_2$?
* $\alpha_0$: Este simplemente se interpreta como el intercepto, donde, si todas las variables son 0, este sería el valor base.
* $\alpha_1$: Podemos notar que en el gráfico entre `ShareA` y `VoteA`, sí existe una relación lineal con cierta dispersión entre algunos datos, pero la forma en general es lineal y este valor describe la pendiente, en donde por cada cambio en `ShareA`, `VoteA` aumentará 0.27 unidades.
* $\alpha_2$: Este valor es cercano a 0, por lo que se entiende que en general la variable `ExpendA` no es influyente para la regresión lineal, no tiene relevancia o relación.

## d.
$voteA=\alpha_0+\alpha_1*shareA+\alpha_2*expendB$

Fórmulas Directas

In [ ]:
x_1 = df["shareA"].to_numpy()
x_2 = df["expendB"].to_numpy()
y = df["voteA"].to_numpy()

In [ ]:
model = RegresionLineal2Vars(y, x_1, x_2)
model.fit()
model.name_vars("voteA", "shareA", "expendB")

In [ ]:
print(model)

$voteA = 40.9158 + 0.2739*shareA + 0*expendB$

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(x_1, x_2, y, c='blue', marker='o', label='Actual data')
x1_surf, x2_surf = np.meshgrid(np.linspace(x_1.min(), x_1.max(), 100), np.linspace(x_2.min(), x_2.max(), 100))
y_surf = model.alpha0 + model.alpha1 * x1_surf + model.alpha2 * x2_surf
ax.plot_surface(x1_surf, x2_surf, y_surf, color='red', alpha=0.5, label='Regression Plane')

ax.set_xlabel('shareA')
ax.set_ylabel('expendB')
ax.set_zlabel('voteA')
ax.set_title('voteA vs shareA and expendA')

ax.legend()
plt.show()

In [ ]:
print("R^2:", R2(y, model.predict(x_1, x_2)))

In [ ]:
model.print_efficiency_details()

In [ ]:
df["artificial_shareA"] = df["expendA"]/(df["expendA"]+df["expendB"])*100
df["diff"] = df["artificial_shareA"] - df["shareA"]
df["diff"] = df["diff"].map(lambda x: "Identical" if float(x) < 0.01 else "Different")
df["diff"].value_counts()

¿Qué nos dicen los respectivos coeficientes $\alpha_0$, $\alpha_1$ y $\alpha_2$?
* $\alpha_0$: Este simplemente se interpreta como el intercepto, donde, si todas las variables son 0, este sería el valor base.
* $\alpha_1$: Podemos notar que en el gráfico entre `ShareA` y `VoteA`, sí existe una relación lineal con cierta dispersión entre algunos datos, pero la forma en general es lineal y este valor describe la pendiente, en donde por cada cambio en `ShareA`, `VoteA` aumentará 0.27 unidades.
* $\alpha_2$: Este valor es cercano a 0, por lo que se entiende que en general la variable `ExpendB` no es influyente para la regresión lineal, no tiene relevancia o relación.
<br>En ambos casos, al añadir `ExpendA` o `ExpendB`, no se aportó nada de información, pues la variable `ShareA` ya contiene la información de ambos al ser el porcentaje de votos totales, derivado de la fórmula:
$$share_A=\frac{expend_A}{expend_A+expend_B}*100$$
Lo cual indica que añadir estas variables no solo es redundante, sino que puede causar cierta multicolinealidad.

## e.
$voteA=\alpha_0+\alpha_1*shareA$

Gradiente Descenciente

In [ ]:
x1 = df["shareA"].to_numpy()
Y_datos = df["voteA"].to_numpy()
m = len(x1)

#Definir parámetros
lr = 1e-5
a0, a1 = 0.0, 0.0
epochs = 10_000_000
epsilon = 1e-8

ECM = []
ECMA, ECMB = 20, 0
epocas = 0

#Medición del tiempo
tiempo_inicial = time.time()
tm.start()

while epocas < epochs and np.abs(ECMA - ECMB) > epsilon:
  #Calcular Y del modelo propuesto
  y_obt = a0 + a1 * x1
  ecm = (1/(2*m)) * np.sum((Y_datos - y_obt)**2)
  ECM.append(ecm)

  #Calcular a0, a1
  a0 = a0 - lr * np.sum(y_obt - Y_datos) / m
  a1 = a1 - lr * np.dot((y_obt - Y_datos), x1) / m

  #Guardar los errores presente y pasado
  ECMB = ECMA
  ECMA = ecm
  #Incremento del iterador
  epocas += 1

#Tiempo final, memoria pico y actual usada
tiempo_fin = time.time()
actual, pico = tm.get_traced_memory()
tm.stop()

print(f"voteA = {a0} + {a1}*shareA")
print("R^2:", R2(Y_datos, a0 + a1 * x1))
print(f"Épocas usadas: {epocas}")
print(f"Tiempo: {tiempo_fin - tiempo_inicial} segundos")
print(f"Memoria pico: {pico/10**6} Mb, actual: {actual/10**6} Mb")

plt.figure()
plt.plot(ECM)
plt.title("ECM vs Epochs")
plt.xlabel("Epochs")
plt.ylabel("ECM")
plt.show()

plt.figure()
plt.scatter(x1, Y_datos)
plt.plot(x1, a0 + a1 * x1, color="red")
plt.xlabel("shareA")
plt.ylabel("voteA")
plt.show()

## f.
$voteA=\alpha_0+\alpha_1*expendA$

Gradiente Descenciente

## g.
$voteA=\alpha_0+\alpha_1*shareA+\alpha_2*expendA$

Gradiente Descenciente

In [ ]:
x_1 = df["shareA"].to_numpy()
x_2 = df["expendA"].to_numpy()
y = df["voteA"].to_numpy()

In [ ]:
model = RegresionLineal2VarGradient(y, x_1, x_2)
model.fit(1e-5, 10000000, 1e-8, (0,0,0))
model.name_vars("voteA", "shareA", "expendA")

In [ ]:
print(model)

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(x_1, x_2, y, c='blue', marker='o', label='Actual data')
x1_surf, x2_surf = np.meshgrid(np.linspace(x_1.min(), x_1.max(), 100), np.linspace(x_2.min(), x_2.max(), 100))
y_surf = model.alpha0 + model.alpha1 * x1_surf + model.alpha2 * x2_surf
ax.plot_surface(x1_surf, x2_surf, y_surf, color='red', alpha=0.5, label='Regression Plane')

ax.set_xlabel('shareA')
ax.set_ylabel('expendA')
ax.set_zlabel('voteA')
ax.set_title('voteA vs shareA and expendA')

ax.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
plt.plot(model.MSEs)
plt.title("MSE vs Epochs")
plt.xlabel("Epochs")
plt.ylabel("MSE")
plt.show()

In [ ]:
print("R^2:", R2(y, model.predict(x_1, x_2)))

In [ ]:
model.print_efficiency_details()

## h.
$voteA=\alpha_0+\alpha_1*shareA+\alpha_2*expendB$

Gradiente Descenciente

In [ ]:
x_1 = df["shareA"].to_numpy()
x_2 = df["expendB"].to_numpy()
y = df["voteA"].to_numpy()

In [ ]:
model = RegresionLineal2VarGradient(y, x_1, x_2)
model.fit(1e-5, 10000000, 1e-8, (0,0,0))
model.name_vars("voteA", "shareA", "expendB")

In [ ]:
print(model)

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(x_1, x_2, y, c='blue', marker='o', label='Actual data')
x1_surf, x2_surf = np.meshgrid(np.linspace(x_1.min(), x_1.max(), 100), np.linspace(x_2.min(), x_2.max(), 100))
y_surf = model.alpha0 + model.alpha1 * x1_surf + model.alpha2 * x2_surf
ax.plot_surface(x1_surf, x2_surf, y_surf, color='red', alpha=0.5, label='Regression Plane')

ax.set_xlabel('shareA')
ax.set_ylabel('expendB')
ax.set_zlabel('voteA')
ax.set_title('voteA vs shareA and expendB')

ax.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
plt.plot(model.MSEs)
plt.title("MSE vs Epochs")
plt.xlabel("Epochs")
plt.ylabel("MSE")
plt.show()

In [ ]:
print("R^2:", R2(y, model.predict(x_1, x_2)))

In [ ]:
model.print_efficiency_details()